# A Machine Learning Approach to Predicting GDP Growth Using World Bank Development Indicators

**Student:** Muhammad Wasal Imtiaz (24077342)  
**Module:** 7PAM2002 Data Science Project  
**University of Hertfordshire — MSc Data Science**

---

## Research Questions
1. Which machine learning models best predict GDP growth using socio-economic and development indicators from the World Bank Open Data platform?
2. How do linear and non-linear machine learning models differ in predictive performance and generalisation when forecasting GDP growth across countries and time periods?
3. Which socio-economic indicators contribute most to GDP growth prediction, and how consistent are these feature importance patterns across different models?

## Indicators Used
| Indicator | Code | Role |
|---|---|---|
| GDP growth (annual %) | NY.GDP.MKTP.KD.ZG | **Target** |
| Inflation, consumer prices (annual %) | FP.CPI.TOTL.ZG | Macroeconomic stability |
| Unemployment (% of labour force) | SL.UEM.TOTL.ZS | Labour market |
| Internet users (% of population) | IT.NET.USER.ZS | Technology adoption |
| Population growth (annual %) | SP.POP.GROW | Demographics |
| Education expenditure (% of GDP) | SE.XPD.TOTL.GD.ZS | Human capital |
| Energy use (kg oil eq. per capita) | EG.USE.PCAP.KG.OE | Economic activity |

## Models
1. **Ridge Regression** (linear baseline)
2. **Random Forest Regressor** (non-linear, ensemble)
3. **Gradient Boosting Regressor** (optimised, ensemble)

---
## 1. Setup & Data Loading

In [1]:

# Mount Google Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# FILE PATHS — edit BASE if your files are in a subfolder
# ============================================================
BASE = "/content/drive/MyDrive/"

PATHS = {
    "GDP_Growth":      BASE + "API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_26.csv",
    "Inflation":       BASE + "API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_32.csv",
    "Unemployment":    BASE + "API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_44.csv",
    "Internet_Users":  BASE + "API_IT.NET.USER.ZS_DS2_en_csv_v2_1228.csv",
    "Pop_Growth":      BASE + "API_SP.POP.GROW_DS2_en_csv_v2_322.csv",
    "Education_Exp":   BASE + "API_SE.XPD.TOTL.GD.ZS_DS2_en_csv_v2_236.csv",
    "Energy_Use":      BASE + "API_EG.USE.PCAP.KG.OE_DS2_en_csv_v2_2093.csv",
}

import os
for name, path in PATHS.items():
    print(f"{name:20s} -> exists: {os.path.exists(path)}")

GDP_Growth           -> exists: True
Inflation            -> exists: True
Unemployment         -> exists: True
Internet_Users       -> exists: True
Pop_Growth           -> exists: True
Education_Exp        -> exists: True
Energy_Use           -> exists: True


In [3]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', 20)
print("All imports successful.")

All imports successful.


In [4]:
# ============================================================
# HELPER: Load & reshape a World Bank indicator CSV (wide -> long)
# ============================================================
def load_worldbank_indicator(csv_path: str, value_name: str) -> pd.DataFrame:
    """Read a World Bank CSV (4 header rows), drop metadata cols, reshape wide->long."""
    df = pd.read_csv(csv_path, skiprows=4)
    df = df.drop(columns=["Indicator Name", "Indicator Code", "Unnamed: 69"], errors="ignore")
    df_long = df.melt(
        id_vars=["Country Name", "Country Code"],
        var_name="Year",
        value_name=value_name
    )
    df_long["Year"] = pd.to_numeric(df_long["Year"], errors="coerce")
    return df_long

# Load all 7 indicators
datasets = {}
for name, path in PATHS.items():
    datasets[name] = load_worldbank_indicator(path, name)
    print(f"Loaded {name}: {datasets[name].shape}")

Loaded GDP_Growth: (17290, 4)
Loaded Inflation: (17290, 4)
Loaded Unemployment: (17290, 4)
Loaded Internet_Users: (17290, 4)
Loaded Pop_Growth: (17290, 4)
Loaded Education_Exp: (17290, 4)
Loaded Energy_Use: (17290, 4)


In [5]:
# ============================================================
# MERGE all indicators into one panel dataset (Country x Year)
# ============================================================
keys = ["Country Name", "Country Code", "Year"]

# Start with GDP (target)
indicator_names = list(PATHS.keys())
data = datasets[indicator_names[0]].copy()

# Merge remaining indicators
for name in indicator_names[1:]:
    data = data.merge(datasets[name], on=keys, how="inner")

print(f"Raw merged dataset: {data.shape}")
print(f"Year range: {data['Year'].min():.0f} — {data['Year'].max():.0f}")
data.head()

Raw merged dataset: (17290, 10)
Year range: 1960 — 2024


,Country Name,Country Code,Year,GDP_Growth,Inflation,Unemployment,Internet_Users,Pop_Growth,Education_Exp,Energy_Use
0,Aruba,ABW,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN
